# Lesson 13B: Diffusion Models - Practical

Train simplified diffusion model on 2D data.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(42)
print('✅ Loaded!')

In [ ]:
class SimpleDiffusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_embed = nn.Linear(1, 32)
        self.net = nn.Sequential(nn.Linear(34, 64), nn.ReLU(), nn.Linear(64, 2))
    
    def forward(self, x, t):
        t_emb = self.time_embed(t.unsqueeze(-1))
        return self.net(torch.cat([x, t_emb], dim=-1))

model = SimpleDiffusion()
print('✅ Model defined')

In [ ]:
# Train
X, _ = make_moons(n_samples=1000, noise=0.1, random_state=42)
X = torch.FloatTensor(X)

def get_beta_schedule(T=100):
    return torch.linspace(0.0001, 0.02, T)

def forward_diffusion(x0, t, betas):
    alpha = 1 - betas
    alpha_bar = torch.cumprod(alpha, dim=0)
    alpha_t = alpha_bar[t]
    noise = torch.randn_like(x0)
    x_t = torch.sqrt(alpha_t) * x0 + torch.sqrt(1 - alpha_t) * noise
    return x_t, noise

betas = get_beta_schedule()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(200):
    batch = X[torch.randint(0, len(X), (64,))]
    t = torch.randint(0, 100, (64,))
    x_noisy, noise = forward_diffusion(batch, t, betas)
    noise_pred = model(x_noisy, t.float())
    loss = nn.MSELoss()(noise_pred, noise)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 50 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')
print('✅ Diffusion model trained!')